# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset title and description
print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
# Print publication date, license, version
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")
print(f"Version: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We review the record sets, their `@id`s and fields. All references below are to `@id` values as required.

In [ ]:
# List record sets in the dataset, enumerate their @id and their fields/columns
record_set_objs = dataset.metadata.recordSets  # This is a list of RecordSet objects

print("Available Record Sets:")
record_set_ids = []  # Collect @id's for later

for rs in record_set_objs:
    print(f"  Record Set Name: {rs.name}")
    print(f"    @id: {rs.id}")
    record_set_ids.append(rs.id)
    # List the fields and columns for this record set
    print(f"    Fields and Columns (with @id):")
    for field in rs.fields:
        print(f"      Field: {field.name}, @id: {field.id}, DataType: {field.dataType}")
    for col in rs.columns:
        print(f"      Column: {col.name}, @id: {col.id}, DataType: {col.dataType}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect records from each record set into pandas DataFrames.
# All references are by @id.

dataframes = {}
for record_set_id in record_set_ids:
    # Each 'record_set_id' is the @id for a Croissant recordSet
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Number of records: {len(df)}")
    print(f"  Columns (@id): {df.columns.tolist()}\n")

# For illustration, select the first record set
example_record_set_id = record_set_ids[0]
print(f"Columns for example record set ('{example_record_set_id}'):")
print(dataframes[example_record_set_id].columns.tolist())

print("\nFirst five records:")
dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# ------ EDA setup. Refer to fields and columns by their @id. ------
# For this dataset, let's select a numeric field, e.g., percent_coverage (@id) or MW (@id), from earlier overview.

# Please replace these with the actual @id of a numeric field and a group field from your schema.
# For demonstration, let's search for a numeric field and a group field:

example_df = dataframes[example_record_set_id]
numeric_field_id = None
group_field_id = None

for col in example_df.columns:
    # Try to infer numeric field
    try:
        # Try convert to numeric to check
        pd.to_numeric(example_df[col].dropna().iloc[:5])
        if numeric_field_id is None:
            numeric_field_id = col
    except Exception:
        # Not numeric
        if group_field_id is None:
            group_field_id = col

print(f"Using column '{numeric_field_id}' as the numeric field for EDA.")
print(f"Using column '{group_field_id}' as the group field (if any).\n")

# Remove rows with missing values in our numeric field
eda_df = example_df.copy()
eda_df = eda_df[eda_df[numeric_field_id].notnull()]

# Convert the field to numeric (if not already numeric)
eda_df[numeric_field_id] = pd.to_numeric(eda_df[numeric_field_id], errors='coerce')
eda_df = eda_df[eda_df[numeric_field_id].notnull()]

# Filtering records where numeric_field > threshold
threshold = eda_df[numeric_field_id].median()
filtered_df = eda_df[eda_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped analysis by another field, if suitable
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped data (mean of {numeric_field_id}) by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(eda_df[numeric_field_id], bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouped_df exists, plot top 10 mean values
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df.head(10), palette="viridis")
    plt.title(f"Top 10 '{group_field_id}' by mean '{numeric_field_id}'")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=30, ha='right')
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We used `mlcroissant` to load the Croissant schema-defined dataset, referenced each record set and field by its `@id`, and demonstrated how to explore both structure and data.
- Sample EDA was performed on a numeric field, including outlier filtering and normalization.
- The dataset includes information about protein mass spectrometry measurements from extracellular vesicles from human mast cells.

Further analysis can be performed as per research needs, including comparison of protein groups or deeper domain-specific preprocessing based on column/field semantics.